# Kidney Stones Dataset Preprocessing

This notebook is dedicated to preprocess the given dataset into a required format/structure and divide it into train/val/test splits.


# Google Colab only

### Download required packages

In [ ]:
!pip install -r https://raw.githubusercontent.com/pmalesa/lesion_detector/main/notebooks/requirements.txt

### Mount DeepLesion images and checkpoints from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content

# remove existing link if any
!rm -rf data/kidney_stones

!mkdir -p data
!ln -s /content/drive/MyDrive/datasets/kidney_stones/data/kidney_stones data/kidney_stones
!ls -l data

# Import all packages

In [ ]:
# General packages
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.typing import NDArray
from typing import Any
from PIL import Image
import shutil
import random
from pathlib import Path
import yaml
import cv2

# Set paths to kidney stones images and metadata

## Paths to unprocessed data

In [ ]:
# Google Colab
kidney_stones_images_path_unprocessed = Path("data/kidney_stones/")
kidney_stones_labels_path_unprocessed = Path("data/kidney_stones/")

In [ ]:
# Local
kidney_stones_images_path_unprocessed = Path("../data/kidney_stones/")
kidney_stones_labels_path_unprocessed = Path("../data/kidney_stones/")

## Paths to processed data

In [ ]:
kidney_stones_data_dir = Path("data/kidney_stones/")
kidney_stones_images_path_processed = kidney_stones_data_dir / "kidney_stones_processed_uint8/images"
kidney_stones_labels_path_processed = kidney_stones_data_dir / "kidney_stones_processed_uint8/labels"

## Paths to target split directory

In [ ]:
current_seed = 42  #   [42, 314. 666]
seed_split_map = {
    42: "1",
    314: "2",
    666: "3"
}

target_yolo_dir = kidney_stones_data_dir / f"kidney_stones_yolo_split_{seed_split_map[current_seed]}"
target_fasterrcnn_dir = kidney_stones_data_dir / f"kidney_stones_fasterrcnn_split_{seed_split_map[current_seed]}"

IMAGE_SIZE = 512
random.seed(current_seed)

print(f"INFO: Current seed: {current_seed} / Current split: {seed_split_map[current_seed]}")

# Process the kidney stones dataset

In [ ]:
'''
Preprocess the kidney stones dataset by:
    1) Load all the images and labels from original kidney stones dataset (initially divided into train/val/test split),
    2) Resize each image to 512x512 uint8 grayscale PNG format (original images were JPGs),
    3) Save each image to "kidney_stones_processed_uint8/images/images_xxxx.png" path,
    4) Save each label in cxcywh normalized bbox format to "kidney_stones_processed_uint8/labels/label_xxxx.txt" (original format was cxcywh normalized already),
    5) Result processed kidney stone dataset layout:

        |-kidney_stones_processed_uint8
        |----images
            |----image_0001.png
            |----image_0002.png
                      (...)
        |----labels
            |----image_0001.png
            |----image_0001.png
                      (...)
'''

# Clear the existing directory with preprocessed images
if kidney_stones_images_path_processed.exists() and kidney_stones_images_path_processed.is_dir():
    shutil.rmtree(kidney_stones_images_path_processed)
kidney_stones_images_path_processed.mkdir(parents=True, exist_ok=True)

if kidney_stones_labels_path_processed.exists() and kidney_stones_labels_path_processed.is_dir():
    shutil.rmtree(kidney_stones_labels_path_processed)
kidney_stones_labels_path_processed.mkdir(parents=True, exist_ok=True)

(kidney_stones_images_path_processed).mkdir(parents=True, exist_ok=True)
(kidney_stones_labels_path_processed).mkdir(parents=True, exist_ok=True)

# 1) Collect all samles (image + label)
samples = []
for split in ["train", "valid", "test"]:
    image_dir = kidney_stones_images_path_unprocessed / split / "images"
    label_dir = kidney_stones_labels_path_unprocessed / split / "labels"

    if not image_dir.exists():
        continue

    for image_path in image_dir.glob("*"):
        label_path = label_dir / (image_path.stem + ".txt")
        if label_path.exists():
            samples.append((image_path, label_path))

print(f"Total (image, label) samples found: {len(samples)}")

# 2) Load each image save it in 512x512 grayscale PNG format (+ rename)
for i, (unprocessed_image_path, unprocessed_label_path) in enumerate(samples):
    image_str_number = f"{i + 1:05d}"
    processed_image_path = kidney_stones_images_path_processed / f"image_{image_str_number}.png"
    processed_label_path = kidney_stones_labels_path_processed / f"image_{image_str_number}.txt"

    image = Image.open(unprocessed_image_path).convert("L") # Convert to grayscale (1 channel)
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)

    image.save(processed_image_path, format="PNG")
    shutil.copy(unprocessed_label_path, processed_label_path)

    print(f"\rProcessed: {i + 1} / {len(samples)}", end="", flush=True)
    
print()

# 3) Verify the results
for i in range(len(samples)):
    image_str_number = f"{i + 1:05d}"
    processed_image_path = kidney_stones_images_path_processed / f"image_{image_str_number}.png"
    processed_label_path = kidney_stones_labels_path_processed / f"image_{image_str_number}.txt"

    image = Image.open(processed_image_path)
    arr = np.array(image)
    if arr.shape != (512, 512) or arr.dtype != np.uint8:
        raise ValueError(f"ERROR: Not all images were processed successfully!")
    print(f"\rVerified: {i + 1} / {len(samples)}", end="", flush=True)

print()
print(f"\nSUCCES: All images were preprocessed correctly.")

# Create a single train/val/test data split

## YOLO format

In [ ]:
'''
0...K-1 classes' labels in .txt files

Create a single kidney stones dataset split by:
    1) Load all image paths and label paths into arrays
    2) Split into train, val and test sets
    3) Copy the files to the correct target subdirectories and create kedney_stones.yaml

    Target layout:
    
        |-kidney_stones_yolo_split_X
        |----images
            |----train
                |----image_0001.png
                |----image_0002.png
                      (...)
            |----val
            |----test
        |----labels
            |----train
                |----image_0001.txt
                |----image_0002.txt
                      (...)
            |----val
            |----test
        |----kidney_stones.yaml
'''

# Clear the existing directory with preprocessed images
if target_yolo_dir.exists() and target_yolo_dir.is_dir():
    shutil.rmtree(target_yolo_dir)

splits = ["train", "val", "test"]
for split in splits:
    (target_yolo_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (target_yolo_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

# 1) Collect all images with annotated lesions
image_dir = kidney_stones_images_path_processed
label_dir = kidney_stones_labels_path_processed
annotated_images = sorted([img for img in image_dir.glob("*.png") if (label_dir / (img.stem + ".txt")).exists()]) # sort images
random.seed(current_seed)
random.shuffle(annotated_images)

# 2) Split into train, val and test sets
n_total = len(annotated_images)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)

train_images = annotated_images[:n_train]
val_images = annotated_images[n_train:n_train + n_val]
test_images = annotated_images[n_train + n_val:]

splits_map = {
    "train": train_images,
    "val": val_images,
    "test": test_images
}

# 3) Copy image files
i = 1
for split, images in splits_map.items():
    for image_path in images:
        label_path = label_dir / (image_path.stem + ".txt")
        shutil.copy2(image_path, target_yolo_dir / "images" / split / image_path.name)
        shutil.copy2(label_path, target_yolo_dir / "labels" / split / label_path.name)
        print(f"\rCopied: {i} / { n_total}", end="", flush=True)
        i += 1

print()
print(f"\nSplit done! Total = {n_total}")

# 4) Generate deeplesion.yaml
dataset_root = os.path.abspath(str(target_yolo_dir))
kidney_stones_yaml = {
    "path": dataset_root,
    "train": os.path.join(dataset_root, "images/train"),
    "val": os.path.join(dataset_root, "images/val"),
    "test": os.path.join(dataset_root, "images/test"),
    "nc": 1,
    "names": [
        "kidney_stone"
    ]
}

with open(target_yolo_dir / "kidney_stones.yaml", "w") as f:
    yaml.dump(kidney_stones_yaml, f)

## Faster R-CNN format

In [ ]:
'''
1...K classes' labels in .txt files

Create a single kidney stones dataset split by:
    1) Load all image paths and label paths into arrays
    2) Split into train, val and test sets
    3) Copy the files to the correct target subdirectories and create kedney_stones.yaml
    4) Modify each .txt file by converting from cxcywh normalized to xyxy unnormalized.
        4.1) Load image and its width and height
        4.2) Load the cxcywh normalized values
        4.3) Convert each line to: "{cls + 1} x_min y_min x_max y_max"
        4.4) Save as as the target label textfile

    Target layout:
    
        |-kidney_stones_fasterrcnn_split_X
        |----images
            |----train
                |----image_0001.png
                |----image_0002.png
                      (...)
            |----val
            |----test
        |----labels
            |----train
                |----image_0001.txt
                |----image_0002.txt
                      (...)
            |----val
            |----test
'''

# Clear the existing directory with preprocessed images
if target_fasterrcnn_dir.exists() and target_fasterrcnn_dir.is_dir():
    shutil.rmtree(target_fasterrcnn_dir)

splits = ["train", "val", "test"]
for split in splits:
    (target_fasterrcnn_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (target_fasterrcnn_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

# 1) Collect all images with annotated lesions
image_dir = kidney_stones_images_path_processed
label_dir = kidney_stones_labels_path_processed
annotated_images = sorted([img for img in image_dir.glob("*.png") if (label_dir / (img.stem + ".txt")).exists()]) # sort images
random.seed(current_seed)
random.shuffle(annotated_images)

# 2) Split into train, val and test sets
n_total = len(annotated_images)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)

train_images = annotated_images[:n_train]
val_images = annotated_images[n_train:n_train + n_val]
test_images = annotated_images[n_train + n_val:]

splits_map = {
    "train": train_images,
    "val": val_images,
    "test": test_images
}

# 3) Copy image files
i = 1
for split, images in splits_map.items():
    for image_path in images:
        label_path = label_dir / (image_path.stem + ".txt")

        target_images_path = target_fasterrcnn_dir / "images" / split / image_path.name
        target_label_path = target_fasterrcnn_dir / "labels" / split / label_path.name

        shutil.copy2(image_path, target_images_path)
        
        # Convert cxcywh normalized to xyxy unnormalized (and klass from 0...K-1 to 1...K)
        with Image.open(image_path) as img:
            W, H = img.size

        new_lines = []
    
        with open(label_path, "r") as f:
            for line in f:
                cls, cx, cy, w, h = map(float, line.strip().split())

                # class 0..K-1 -> 1..K
                cls = int(cls) + 1

                # denormalize
                cx *= W
                cy *= H
                w *= W
                h *= H

                # convert to xyxy
                x1 = cx - w / 2
                y1 = cy - h / 2
                x2 = cx + w / 2
                y2 = cy + h / 2

                # convert to integers
                x1 = int(round(x1))
                y1 = int(round(y1))
                x2 = int(round(x2))
                y2 = int(round(y2))

                # clamp to image bounds
                x1 = max(0, min(W - 1, x1))
                y1 = max(0, min(H - 1, y1))
                x2 = max(0, min(W - 1, x2))
                y2 = max(0, min(H - 1, y2))

                new_lines.append(f"{cls} {x1} {y1} {x2} {y2}\n")

        with open(target_label_path, "w") as f:
            f.writelines(new_lines)

        print(f"\rCopied: {i} / {n_total}", end="", flush=True)
        i += 1

print()
print(f"\nSplit done! Total = {n_total}")